# Recommended SAMMD workflow

This notebook runs the usual SAMMD steps without needing OpenMM, OpenFF, RDKit, PACKMOL, or long calculations. It checks a YAML template, makes a fast reproducible build plan, writes the default output files, and shows what is inside them.

If you run SAMMD in a CUDA-labeled pixi environment, you can also use `--full` to write files for OpenMM and OpenFF: `solvated_system.cif`, `interchange.json`, `system.xml`, and `anchor_metadata.json`. Run `nvidia-smi` first, then choose an environment such as `cuda-12-4`, `cuda-12-6`, or `cuda-13-0`. `interchange.json` stores the OpenFF `Interchange`; `system.xml` is an OpenMM file. SAMMD prepares the input files, while OpenMM runs minimization, equilibration, production, trajectories, and reporters.

## Create a template configuration

The CLI command `sammd init -o sammd.yaml` writes the same template. Here we write it directly so the notebook stays self-contained.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from sammd.core.config import CONFIG_TEMPLATE

workspace = TemporaryDirectory()
workdir = Path(workspace.name)
config_path = workdir / "sammd.yaml"
output_dir = workdir / "outputs"
config_path.write_text(CONFIG_TEMPLATE, encoding="utf-8")

print(f"Wrote template config to {config_path}")

## Validate, load, and build a plan

`load_config()` checks the YAML with the Pydantic schema. `build_system()` creates the same build plan every time for the same configuration.

In [ ]:
from sammd import build_system, load_config

config = load_config(config_path)
plan = build_system(config, output_dir=output_dir)

print("Configuration validated")
print(f"Full OpenMM construction available: {plan.full_construction_available}")

## Write default output files

The default lightweight build writes exactly three output files: `sam_grafting_density.cif`, `build_summary.json`, and `resolved_config.yaml`. In this mode, `sam_grafting_density.cif` is a visual smoke test for the Pd slab and SAM sulfur anchor positions.

In [ ]:
topology_path = plan.write_topology_cif(overwrite=True)
build_summary_path = plan.write_build_summary(overwrite=True)
resolved_config_path = plan.write_resolved_config(overwrite=True)

for path in (topology_path, build_summary_path, resolved_config_path):
    print(f"Wrote {path.name}: {path}")

## Inspect default outputs

These values help you check the surface, the adjusted box size, how many SAM molecules were planned, how many solution molecules were counted, and which files were written. Open `sam_grafting_density.cif` in a molecule viewer such as PyMOL to inspect the Pd slab and sulfur anchor atoms. This smoke test lets you check slab geometry, three-fold hollow-site placement, and grafting density before exporting full SAM, solvent, and reactant coordinates.

In [ ]:
summary = {
    "surface": f"{plan.slab.metal}({plan.slab.facet})",
    "adjusted_lateral_size_nm": tuple(round(value, 3) for value in plan.slab.lateral_size_nm),
    "binding_site_count": len(plan.binding_sites),
    "sam_placement_count": len(plan.sam_placements.placements),
    "sam_sites_per_side": plan.sam_placements.selected_sites_per_side,
    "solution_counts": plan.solution.molecule_counts,
}

for key, value in summary.items():
    print(f"{key}: {value}")

print()
for path in (topology_path, build_summary_path, resolved_config_path):
    print(f"{path.name}: exists={path.exists()} size={path.stat().st_size} bytes")

## Optional OpenMM/OpenFF files

By default, this notebook does not write `solvated_system.cif`, `interchange.json`, `system.xml`, or `anchor_metadata.json`. To create full SAM, solvent, and reactant coordinates, run the CLI with `--full` in a CUDA-labeled pixi environment and inspect `solvated_system.cif`. In this version, `--full` does not work with configurations that include salt.

SAMMD writes `interchange.json` with `Interchange.model_dump_json` and reads it back with `Interchange.model_validate_json`. OpenFF Interchange is still pre-1.0, so JSON files may not work across all OpenFF Interchange versions.

In [ ]:
import subprocess
import sys

RUN_BACKEND_EXPORT = False  # Set True only in a CUDA-labeled pixi environment.
SAMMD_PIXI_ENV = "cuda-12-4"  # Change after checking nvidia-smi.
backend_output_dir = workdir / "backend_outputs"

backend_command = [
    "sammd",
    "build",
    str(config_path),
    "--output-dir",
    str(backend_output_dir),
    "--overwrite",
    "--full",
]

if RUN_BACKEND_EXPORT:
    subprocess.run(backend_command, check=True)
else:
    print(f"Skipped backend export. Run this notebook with `pixi run -e {SAMMD_PIXI_ENV} jupyter lab` and set RUN_BACKEND_EXPORT = True.")
    print("Equivalent CLI command:")
    print(f"pixi run -e {SAMMD_PIXI_ENV} " + " ".join(backend_command))

## Use these files with OpenMM

After `--full` runs, use the output files in your own OpenMM Python script. Load the OpenFF `Interchange` from `interchange.json` with `Interchange.model_validate_json`. Then create an `OpenMM System` with `interchange.to_openmm()`, load the coordinates from `solvated_system.cif`, and run an OpenMM `Simulation` with the OpenMM Python API. You can also use `system.xml` as an OpenMM file.

This notebook does not create or run an OpenMM simulation. SAMMD prepares the files, and OpenMM runs minimization, equilibration, production, trajectories, and reporters.

In [ ]:
if RUN_BACKEND_EXPORT:
    from openff.interchange import Interchange

    interchange_path = backend_output_dir / "interchange.json"
    interchange = Interchange.model_validate_json(interchange_path.read_text(encoding="utf-8"))
    openmm_system = interchange.to_openmm()

    for name in ("sam_grafting_density.cif", "solvated_system.cif", "interchange.json", "system.xml", "anchor_metadata.json"):
        path = backend_output_dir / name
        print(f"{name}: exists={path.exists()} size={path.stat().st_size if path.exists() else 0} bytes")

    print(f"OpenMM System particles from Interchange: {openmm_system.getNumParticles()}")
else:
    print("Skipped Interchange reload because RUN_BACKEND_EXPORT is False.")

## Cleanup

The temporary directory can be removed when you are done inspecting generated files. Comment this out if you want to keep `sam_grafting_density.cif`.

In [ ]:
workspace.cleanup()